In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login, delete_repo
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier

MODEL_ID = "andreamoccia/BuffettBot"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [4]:
# Configure FP8 dynamic post-training quantization
recipe = QuantizationModifier(
    targets="Linear",          # Quantize all fully-connected layers
    scheme="FP8_DYNAMIC",      # 8-bit float with runtime scale calculation (no calibration needed)
    ignore=["lm_head"]         # Keep output layer in full precision for better accuracy
)

# Apply quantization in a single pass (no fine-tuning)
oneshot(model=model, recipe=recipe)

# Save the quantized model
SAVE_DIR = MODEL_ID.split("/")[1] + "-FP8-Dynamic"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

2025-11-20T00:52:28.235165+0000 | reset | INFO - Compression lifecycle reset
2025-11-20T00:52:28.240435+0000 | _create_default_logger | INFO - Logging all LLM Compressor modifier-level logs to sparse_logs/20-11-2025_00.52.28.log
2025-11-20T00:52:28.241258+0000 | from_modifiers | INFO - Creating recipe from modifiers
2025-11-20T00:52:28.299009+0000 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2025-11-20T00:52:28.299702+0000 | IndependentPipeline | INFO - Inferred `DataFreePipeline` for `QuantizationModifier`


Updating global scales: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 448/448 [00:00<00:00, 943297.29it/s]
Fusing global scales: 1415it [00:00, 932140.75it/s]
Calibrating weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 448/448 [00:00<00:00, 3268.21it/s]

2025-11-20T00:52:48.244744+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers


2025-11-20T00:53:03.992425+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2025-11-20T00:53:04.022939+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 448it [00:18, 23.64it/s]


('BuffettBot-FP8-Dynamic/tokenizer_config.json',
 'BuffettBot-FP8-Dynamic/special_tokens_map.json',
 'BuffettBot-FP8-Dynamic/chat_template.jinja',
 'BuffettBot-FP8-Dynamic/vocab.json',
 'BuffettBot-FP8-Dynamic/merges.txt',
 'BuffettBot-FP8-Dynamic/added_tokens.json',
 'BuffettBot-FP8-Dynamic/tokenizer.json')

In [11]:
hf_key = "***"
repo_id = MODEL_ID+"-FP8-Dynamic"

In [12]:
delete_repo(
    repo_id=repo_id,
    token=hf_key,
    missing_ok = True
)

In [ ]:
%%capture
model.push_to_hub(repo_id, commit_message="Add FP8 quantized model", private=True)
tokenizer.push_to_hub(repo_id, commit_message="Add tokenizer", private=True)

2025-11-20T00:57:28.050725+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.
